
# Statistical Forecasting Models

This notebook evaluates statistical time-series models for one-month-ahead bilateral trade-flow forecasting. ETS and ARIMA/SARIMA are compared against the naive baselines using the same eligible series and March 2026 test observations.

All model selection and fitting use data available through February 2026. Forecasts are evaluated using the shared metrics defined in `src/evaluation.py`.



## 1. Load Forecasting Data

Load the prepared training data and held-out March 2026 observations for each country.


In [13]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_loading import (
    load_forecasting_data,
    SERIES_KEYS,
    TARGET,
)
from src.evaluation import evaluate_forecasts, save_model_results

import numpy as np
import pandas as pd

training_data, test_data = load_forecasting_data()

for country in training_data:
    print(
        f"{country}: {len(training_data[country]):,} training rows, "
        f"{len(test_data[country]):,} test series"
    )

South Korea: 2,374,832 training rows, 27,823 test series
Slovakia: 2,826,210 training rows, 21,973 test series
USA: 3,880,587 training rows, 35,843 test series


## 3. ETS Forecasts

Fit an exponential smoothing model with an additive trend to each selected series. Models use training observations through February 2026 to forecast March 2026.

In [14]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ets_predictions = {}
ets_failures = {}

for country, train_df in training_data.items():
    test_df = test_data[country]
    country_predictions = []
    country_failures = []

    test_lookup = test_df.set_index(SERIES_KEYS)["Value"]
    grouped = train_df.groupby(SERIES_KEYS, sort=False, observed=True)

    for keys, group in grouped:
        group = group.sort_values("Month")
        y = group[TARGET].astype(float).to_numpy()
        key = keys if isinstance(keys, tuple) else (keys,)
        
        try:
            fitted = ExponentialSmoothing(
            y,
            trend="add",
            seasonal=None,
            initialization_method="estimated",
            ).fit(optimized=True)

            prediction = float(fitted.forecast(1)[0])

        except (ValueError, np.linalg.LinAlgError) as exc:
            country_failures.append({
                **dict(zip(SERIES_KEYS, key)),
                "error": str(exc),
            })
            continue

        country_predictions.append({
            **dict(zip(SERIES_KEYS, key)),
            TARGET: float(test_lookup.loc[key]),
            "ets_prediction": prediction,
        })

    ets_predictions[country] = pd.DataFrame(country_predictions)
    ets_failures[country] = pd.DataFrame(country_failures)

    print(
        f"{country}: {len(ets_predictions[country]):,} successful forecasts; "
        f"{len(ets_failures[country]):,} failed fits"
    )

c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwar

South Korea: 27,823 successful forecasts; 0 failed fits


c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwar

Slovakia: 21,973 successful forecasts; 0 failed fits


c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwar

USA: 35,843 successful forecasts; 0 failed fits


c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:1065: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  res = self._optimize_parameters(res, use_brute, method, minimize_kwargs)


## 4. Evaluate ETS Forecasts

Evaluate the ETS forecasts using MAE, RMSE, and sMAPE, and compare their performance with the naive baselines.

In [15]:
ets_results = []

for country, predictions in ets_predictions.items():
    ets_results.append(
        evaluate_forecasts(
            country=country,
            model_name="ETS",
            actual=predictions["Value"],
            predicted=predictions["ets_prediction"],
            forecast_month="2026-03",
        )
    )

ets_results_df = save_model_results(
    ets_results,
    model_name="Statistical"
)

ets_results_df

Saved model results to: C:\Transfer\InterdisciplinaryProject\results\statistical_results.csv


,Country,Model,Forecast Month,N Series,MAE,RMSE,sMAPE (%)
0,Slovakia,ETS,2026-03,21973,2.543246e+05,2.234920e+06,74.967179
1,South Korea,ETS,2026-03,27823,1.338034e+06,2.460953e+07,97.744936
2,USA,ETS,2026-03,35843,2.743561e+06,2.716795e+07,74.738746


## 5. ARIMA Forecasts

Fit ARIMA(1,1,1) models to the selected series and forecast March 2026. If fitting fails for an individual series, use the last observed value as a fallback.

In [16]:
from statsmodels.tsa.arima.model import ARIMA

arima_predictions = {}
arima_failures = {}

for country, train_df in training_data.items():
    test_df = test_data[country]
    country_predictions = []
    country_failures = []

    test_lookup = test_df.set_index(SERIES_KEYS)[TARGET]
    grouped = train_df.groupby(SERIES_KEYS, sort=False, observed=True)

    for keys, group in grouped:
        group = group.sort_values("Month")
        y = group[TARGET].astype(float).to_numpy()
        key = keys if isinstance(keys, tuple) else (keys,)
        try:
            model = ARIMA(
            y,
            order=(1, 1, 1),
            enforce_stationarity=False,
            enforce_invertibility=False,
            )

            fitted = model.fit()
            prediction = float(fitted.forecast(steps=1)[0])

        except (ValueError, np.linalg.LinAlgError) as exc:
            country_failures.append({
                **dict(zip(SERIES_KEYS, key)),
                "error": str(exc),
            })
            continue

        country_predictions.append({
            **dict(zip(SERIES_KEYS, key)),
            TARGET: float(test_lookup.loc[key]),
            "arima_prediction": prediction,
        })

    arima_predictions[country] = pd.DataFrame(country_predictions)
    arima_failures[country] = pd.DataFrame(country_failures)

    print(
        f"{country}: {len(arima_predictions[country]):,} successful forecasts; "
        f"{len(arima_failures[country]):,} failed fits"
    )

c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning:

South Korea: 27,823 successful forecasts; 0 failed fits


c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning:

Slovakia: 21,973 successful forecasts; 0 failed fits


c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
c:\Transfer\InterdisciplinaryProject\.venv\Lib\site-packages\statsmodels\tsa\statespace\mlemodel.py:737: ConvergenceWarning:

USA: 35,843 successful forecasts; 0 failed fits


## 6.Evaluation and Results

Evaluate the naive and seasonal-naive forecasts using MAE, RMSE, and sMAPE. Save the results in the standardized format defined in `src/evaluation.py` for comparison with other forecasting models.

In [17]:
arima_results = []

for country, predictions in arima_predictions.items():
    arima_results.append(
        evaluate_forecasts(
            country=country,
            model_name="ARIMA(1,1,1)",
            actual=predictions[TARGET],
            predicted=predictions["arima_prediction"],
            forecast_month="2026-03",
        )
    )

arima_results_df = save_model_results(
    arima_results,
    model_name="ARIMA",
)

arima_results_df

Saved model results to: C:\Transfer\InterdisciplinaryProject\results\arima_results.csv


,Country,Model,Forecast Month,N Series,MAE,RMSE,sMAPE (%)
0,Slovakia,"ARIMA(1,1,1)",2026-03,21973,2.418107e+05,2.046319e+06,70.255405
1,South Korea,"ARIMA(1,1,1)",2026-03,27823,1.381687e+06,2.473452e+07,95.018261
2,USA,"ARIMA(1,1,1)",2026-03,35843,5.573713e+06,5.397573e+08,72.317729



## Summary

This notebook evaluated two statistical forecasting approaches for one-month-ahead bilateral trade-flow forecasting: Exponential Smoothing (ETS) and ARIMA. Both models were fitted separately to each eligible product-partner series for South Korea, Slovakia, and the USA.

The models used observations through February 2026 to forecast March 2026. Forecasts were evaluated on the original trade-value scale using MAE, RMSE, and sMAPE, with results saved to the `results/` directory for the final model comparison.

The ETS model included an additive trend without an explicit seasonal component. ARIMA used an ARIMA(1,1,1) specification. Model performance will be compared with the naive baselines and machine learning models at the end of the project.
